## Bước 1: Load và gộp 2 file CSV

Ở bước này, ta đọc hai file dữ liệu Online Retail:

- `online_retail_09_10.csv`
- `online_retail_10_11.csv`

Hai file này chứa dữ liệu giao dịch ở hai giai đoạn khác nhau.  
Vì mục tiêu của project là phân tích giỏ hàng tổng thể, ta sẽ gộp hai file thành một dataframe duy nhất để tiếp tục tiền xử lý dữ liệu.

In [ ]:
import pandas as pd
import numpy as np

# Khai báo đường dẫn/tên file dữ liệu
file_09_10 = "online_retail_09_10.csv"
file_10_11 = "online_retail_10_11.csv"

# Đọc từng file CSV thành dataframe riêng
df_09_10 = pd.read_csv(file_09_10)
df_10_11 = pd.read_csv(file_10_11)

# In kích thước của từng bộ dữ liệu
print("Shape of 2009-2010 data:", df_09_10.shape)
print("Shape of 2010-2011 data:", df_10_11.shape)

# Gộp hai dataframe thành một dataframe chung
df = pd.concat([df_09_10, df_10_11], ignore_index=True)

# In kích thước của dữ liệu sau khi gộp
print("Shape of combined data:", df.shape)

# Hiển thị 5 dòng đầu tiên
df.head()

Shape of 2009-2010 data: (525461, 8)
Shape of 2010-2011 data: (541910, 8)
Shape of combined data: (1067371, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,12/1/2009 7:45,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,12/1/2009 7:45,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,12/1/2009 7:45,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,12/1/2009 7:45,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,12/1/2009 7:45,1.25,13085.0,United Kingdom


## Bước 2: Kiểm tra dữ liệu ban đầu

Ở bước này, ta kiểm tra cấu trúc tổng quan của dữ liệu sau khi gộp.

Mục tiêu của bước này là xem:

- Dữ liệu có bao nhiêu dòng và bao nhiêu cột
- Tên các cột là gì
- Kiểu dữ liệu của từng cột
- Có bao nhiêu giá trị bị thiếu
- Một vài dòng đầu tiên trông như thế nào

Bước này giúp ta hiểu dữ liệu trước khi bắt đầu làm sạch.

In [ ]:
# Xem kích thước dữ liệu: số dòng và số cột
print("Dataset shape:", df.shape)

# Xem tên các cột trong dữ liệu
print("Column names:")
print(df.columns)

# Xem thông tin tổng quan: kiểu dữ liệu và số lượng giá trị không bị thiếu
df.info()

Dataset shape: (1067371, 8)
Column names:
Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country'],
      dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   InvoiceNo    1067371 non-null  object 
 1   StockCode    1067371 non-null  object 
 2   Description  1062989 non-null  object 
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  object 
 5   UnitPrice    1067371 non-null  float64
 6   CustomerID   824364 non-null   float64
 7   Country      1067371 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 65.1+ MB


In [ ]:
# Tính số lượng giá trị bị thiếu
missing_count = df.isnull().sum()

# Tính tỷ lệ phần trăm giá trị bị thiếu
missing_percent = (missing_count / len(df)) * 100

# Tạo bảng tổng hợp
missing_summary = pd.DataFrame({
    "Missing Count": missing_count,
    "Missing Percent (%)": missing_percent
})

# Sắp xếp giảm dần theo tỷ lệ thiếu
missing_summary = missing_summary.sort_values(
    by="Missing Percent (%)",
    ascending=False
)

missing_summary

,Missing Count,Missing Percent (%)
CustomerID,243007,22.766873
Description,4382,0.410541
StockCode,0,0.000000
InvoiceNo,0,0.000000
Quantity,0,0.000000
InvoiceDate,0,0.000000
UnitPrice,0,0.000000
Country,0,0.000000


## Ý nghĩa các cột trong bộ dữ liệu

| **Column** | **Description (Mô tả)** | **Relevance to this study (Vai trò trong nghiên cứu)** |
| --- | --- | --- |
| **InvoiceNo** | Số hóa đơn được gán cho mỗi giao dịch. Nếu bắt đầu bằng chữ `C`, nó biểu thị một giao dịch hủy bỏ. | Được sử dụng làm ID giỏ hàng (basket ID) cho Market Basket Analysis. |
| **StockCode** | Mã sản phẩm/mặt hàng. | Được sử dụng làm định danh mặt hàng chính cho association rule mining. |
| **Description** | Tên sản phẩm hoặc mô tả mặt hàng. | Được sử dụng để giải thích ý nghĩa của các mã sản phẩm trong các luật kết hợp cuối cùng. |
| **Quantity** | Số lượng đơn vị được mua trong mỗi dòng giao dịch. | Được sử dụng để xác định các đơn hàng hợp lệ và các trường hợp trả hàng/hủy đơn có thể xảy ra. |
| **InvoiceDate** | Ngày và giờ giao dịch được tạo. | Được sử dụng để hiểu dữ liệu theo mốc thời gian và phục vụ phân tích theo tháng. |
| **UnitPrice** | Đơn giá của sản phẩm tính bằng bảng Anh (£). | Được sử dụng để tính doanh thu từng dòng và giá trị giỏ hàng. |
| **CustomerID** | Mã định danh khách hàng duy nhất. | Hữu ích cho phân tích ở cấp độ khách hàng, nhưng không bắt buộc đối với luật kết hợp ở cấp độ giỏ hàng. |
| **Country** | Quốc gia của khách hàng. | Hữu ích cho việc lọc theo địa lý hoặc phân tích mô tả theo quốc gia. |


## Bước 3: Chuẩn hóa kiểu dữ liệu

Ở bước này, ta chuyển một số cột về đúng kiểu dữ liệu để thuận tiện cho quá trình xử lý và phân tích.

- `InvoiceDate` được chuyển sang kiểu thời gian (`datetime`) để có thể phân tích theo ngày, tháng, năm.
- `InvoiceNo` được chuyển sang kiểu chuỗi vì đây là mã hóa đơn, không phải biến số để tính toán.
- `StockCode` được chuyển sang kiểu chuỗi vì mã sản phẩm có thể chứa cả chữ và số.
- `CustomerID` được giữ lại nhưng không bắt buộc cho Market Basket Analysis vì ta chủ yếu phân tích theo `InvoiceNo`.

In [ ]:
# Chuyển InvoiceDate sang kiểu datetime để xử lý dữ liệu thời gian
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

# Chuyển InvoiceNo sang kiểu string vì đây là mã hóa đơn/basket ID
df["InvoiceNo"] = df["InvoiceNo"].astype(str)

# Chuyển StockCode sang kiểu string vì mã sản phẩm có thể chứa cả chữ và số
df["StockCode"] = df["StockCode"].astype(str)

# Kiểm tra lại kiểu dữ liệu sau khi chuyển đổi
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   InvoiceNo    1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[ns]
 5   UnitPrice    1067371 non-null  float64       
 6   CustomerID   824364 non-null   float64       
 7   Country      1067371 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 65.1+ MB


## Bước 4: Xử lý dữ liệu bị thiếu

Ở bước này, ta xử lý các giá trị bị thiếu trong dataset.

Từ bước kiểm tra dữ liệu ban đầu:

- `CustomerID` thiếu khoảng 22.77%
- `Description` thiếu khoảng 0.41%

Đối với Market Basket Analysis, `CustomerID` không bắt buộc vì ta phân tích giỏ hàng dựa trên `InvoiceNo`.

Tuy nhiên, `Description` là tên sản phẩm, dùng để giải thích kết quả luật kết hợp. Vì tỷ lệ thiếu của `Description` rất nhỏ, ta sẽ xóa các dòng bị thiếu `Description`.

In [ ]:
# Kiểm tra lại số lượng giá trị bị thiếu trước khi xử lý
missing_before = df.isnull().sum()

print("Missing values before cleaning:")
print(missing_before)

# Xóa các dòng bị thiếu Description vì đây là tên sản phẩm
df = df.dropna(subset=["Description"]).copy()

# Kiểm tra lại số lượng giá trị bị thiếu sau khi xử lý
missing_after = df.isnull().sum()

print("Missing values after cleaning:")
print(missing_after)

# Kiểm tra kích thước dữ liệu sau khi xóa dòng thiếu Description
print("Shape after removing missing Description:", df.shape)

Missing values before cleaning:
InvoiceNo           0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     243007
Country             0
dtype: int64
Missing values after cleaning:
InvoiceNo           0
StockCode           0
Description         0
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     238625
Country             0
dtype: int64
Shape after removing missing Description: (1062989, 8)


## Bước 5: Loại bỏ hóa đơn bị hủy

Trong bộ dữ liệu Online Retail, các hóa đơn bị hủy thường có `InvoiceNo` bắt đầu bằng chữ `C`.

Ví dụ:

- `C489449`
- `C489459`

Các dòng này thể hiện giao dịch bị hủy hoặc trả hàng, không phải giao dịch mua thành công.  
Vì Market Basket Analysis cần phân tích các sản phẩm thực sự được mua cùng nhau, ta sẽ loại bỏ các hóa đơn bị hủy.

In [ ]:
# Đếm số dòng thuộc hóa đơn bị hủy
cancelled_rows = df["InvoiceNo"].str.startswith("C").sum()

print("Number of cancelled invoice rows:", cancelled_rows)

# Loại bỏ các hóa đơn có InvoiceNo bắt đầu bằng chữ C
df = df[~df["InvoiceNo"].str.startswith("C")].copy()

# Kiểm tra kích thước dữ liệu sau khi loại bỏ hóa đơn bị hủy
print("Shape after removing cancelled invoices:", df.shape)

Number of cancelled invoice rows: 19494
Shape after removing cancelled invoices: (1043495, 8)


## Bước 6: Loại bỏ Quantity và UnitPrice không hợp lệ

Trong dữ liệu bán lẻ, các giao dịch hợp lệ phải có:

- Quantity > 0
- UnitPrice > 0

Quantity âm hoặc bằng 0 thường xuất hiện trong các giao dịch trả hàng hoặc dữ liệu bất thường.

UnitPrice bằng 0 hoặc âm không phản ánh giá trị bán hàng thực tế và có thể ảnh hưởng đến việc tính doanh thu hoặc giá trị giỏ hàng.

Do đó, các bản ghi không thỏa mãn điều kiện trên sẽ được loại bỏ.

In [ ]:
# Đếm số dòng có Quantity không hợp lệ
invalid_quantity = (df["Quantity"] <= 0).sum()

# Đếm số dòng có UnitPrice không hợp lệ
invalid_price = (df["UnitPrice"] <= 0).sum()

print("Rows with Quantity <= 0:", invalid_quantity)
print("Rows with UnitPrice <= 0:", invalid_price)

# Chỉ giữ các giao dịch có Quantity và UnitPrice dương
df = df[
    (df["Quantity"] > 0) &
    (df["UnitPrice"] > 0)
].copy()

# Kiểm tra kích thước dữ liệu sau khi làm sạch
print("Shape after removing invalid Quantity and UnitPrice:", df.shape)

Rows with Quantity <= 0: 768
Rows with UnitPrice <= 0: 1825
Shape after removing invalid Quantity and UnitPrice: (1041670, 8)


### Findings

After removing cancelled invoices, there were still 768 rows with `Quantity <= 0` and 1,825 rows with `UnitPrice <= 0`.

These records were removed because they do not represent valid completed purchase transactions.

After this cleaning step, the dataset contains 1,041,670 records and 8 columns.

## Bước 7: Xóa các dòng dữ liệu trùng lặp

Ở bước này, ta kiểm tra và loại bỏ các dòng dữ liệu bị trùng lặp hoàn toàn.

Dòng trùng lặp có nghĩa là tất cả giá trị ở các cột đều giống nhau, bao gồm `InvoiceNo`, `StockCode`, `Description`, `Quantity`, `InvoiceDate`, `UnitPrice`, `CustomerID` và `Country`.

Nếu giữ lại các dòng trùng lặp, tần suất xuất hiện của sản phẩm có thể bị tính cao hơn thực tế, làm ảnh hưởng đến kết quả Market Basket Analysis.

Vì vậy, ta sẽ xóa các dòng trùng lặp trước khi xây dựng giỏ hàng.

In [ ]:
# Đếm số dòng bị trùng lặp hoàn toàn
duplicate_rows = df.duplicated().sum()

print("Number of duplicate rows:", duplicate_rows)

# Xóa các dòng trùng lặp
df = df.drop_duplicates().copy()

# Kiểm tra kích thước dữ liệu sau khi xóa duplicate
print("Shape after removing duplicate rows:", df.shape)

Number of duplicate rows: 33757
Shape after removing duplicate rows: (1007913, 8)


## Bước 8: Làm sạch StockCode và Description

Ở bước này, ta làm sạch hai cột quan trọng nhất cho Market Basket Analysis:

- `StockCode`: mã sản phẩm
- `Description`: tên sản phẩm

Ta cần chuẩn hóa hai cột này vì dữ liệu thực tế có thể bị dư khoảng trắng, viết hoa/thường không thống nhất hoặc có lỗi nhập liệu.

`StockCode` sẽ được dùng để tạo giỏ hàng và khai thác association rules.  
`Description` sẽ được dùng để giải thích ý nghĩa của các mã sản phẩm trong kết quả cuối cùng.

In [ ]:
# Xóa khoảng trắng thừa ở đầu và cuối mã sản phẩm
df["StockCode"] = df["StockCode"].astype(str).str.strip()

# Xóa khoảng trắng thừa và chuyển tên sản phẩm về chữ hoa
df["Description"] = df["Description"].astype(str).str.strip().str.upper()

# Kiểm tra số lượng mã sản phẩm và tên sản phẩm duy nhất
print("Number of unique StockCode:", df["StockCode"].nunique())
print("Number of unique Description:", df["Description"].nunique())

# Hiển thị vài dòng để kiểm tra kết quả
df[["StockCode", "Description"]].head()

Number of unique StockCode: 4916
Number of unique Description: 5356


,StockCode,Description
0,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS
1,79323P,PINK CHERRY LIGHTS
2,79323W,WHITE CHERRY LIGHTS
3,22041,"RECORD FRAME 7"" SINGLE SIZE"
4,21232,STRAWBERRY CERAMIC TRINKET BOX


## Bước 9: Kiểm tra StockCode có độ dài khác 5–6 ký tự hoặc chỉ gồm chữ cái

Quan sát dữ liệu cho thấy phần lớn mã sản phẩm (`StockCode`) thường có độ dài khoảng 5 hoặc 6 ký tự.

Tuy nhiên, một số mã có độ dài khác 5–6 ký tự hoặc chỉ gồm chữ cái có thể là mã đặc biệt, ví dụ:

- Phí vận chuyển
- Giảm giá
- Điều chỉnh thủ công
- Phí hệ thống
- Hàng mẫu

Ở bước này, ta lọc các `StockCode` có định dạng bất thường để kiểm tra thủ công trước khi quyết định loại bỏ.

Lưu ý: bước này chỉ dùng để kiểm tra, chưa xóa dữ liệu.

In [ ]:
# Tạo cột lưu độ dài của StockCode
df["StockCodeLength"] = df["StockCode"].str.len()

# Lọc các StockCode chỉ gồm chữ cái hoặc có độ dài khác 5 và 6 ký tự
unusual_stockcodes = df[
    (df["StockCode"].str.match(r"^[A-Za-z\s]+$", na=False)) |
    (~df["StockCodeLength"].isin([5, 6]))
]

# Tạo bảng tóm tắt các StockCode bất thường
unusual_stockcode_summary = (
    unusual_stockcodes
    .groupby("StockCode")
    .agg(
        Description=("Description", lambda x: " | ".join(sorted(x.dropna().unique())[:5])),
        Frequency=("StockCode", "count"),
        InvoiceCount=("InvoiceNo", "nunique"),
        AvgUnitPrice=("UnitPrice", "mean"),
        MinUnitPrice=("UnitPrice", "min"),
        MaxUnitPrice=("UnitPrice", "max")
    )
    .reset_index()
)

# Thêm cột độ dài StockCode vào bảng tóm tắt
unusual_stockcode_summary["StockCodeLength"] = unusual_stockcode_summary["StockCode"].str.len()

# Sắp xếp theo tần suất giảm dần
unusual_stockcode_summary = unusual_stockcode_summary.sort_values(
    ["Frequency", "StockCode"],
    ascending=[False, True]
)

# Hiển thị 100 StockCode bất thường đầu tiên
unusual_stockcode_summary.head(100)

,StockCode,Description,Frequency,InvoiceCount,AvgUnitPrice,MinUnitPrice,MaxUnitPrice,StockCodeLength
30,POST,POSTAGE,1851,1851,29.466515,0.500,8142.750,4
27,DOT,DOTCOM POSTAGE,1415,1415,218.978170,0.350,4505.170,3
0,15056BL,EDWARDIAN PARASOL BLACK,869,857,6.252348,1.950,13.000,7
28,M,MANUAL,846,784,394.318014,0.060,25111.090,1
9,C2,CARRIAGE,267,267,50.217228,15.000,150.000,2
3,79323LP,LIGHT PINK CHERRY LIGHTS,170,170,6.871765,5.450,13.870,7
1,15056bl,EDWARDIAN PARASOL BLACK,87,87,12.562069,12.460,13.000,7
2,79323GR,GREEN CHERRY LIGHTS,83,83,7.097470,5.450,13.870,7
4,ADJUST,ADJUSTMENT BY JOHN ON 26/01/2010 16 | ADJUSTME...,36,36,247.164722,4.570,5117.030,6
8,BANK CHARGES,BANK CHARGES,34,33,15.271794,0.001,39.240,12


## Bước 10: Loại bỏ các StockCode không phù hợp

Sau khi kiểm tra các `StockCode` có định dạng bất thường, ta xác định được một số mã không đại diện cho sản phẩm thật.

Các mã này bao gồm:

- Phí vận chuyển
- Phí giao hàng online
- Điều chỉnh thủ công
- Giảm giá
- Phí ngân hàng
- Phí nền tảng
- Mã test hệ thống
- Gift voucher
- Điều chỉnh công nợ
- Hàng mẫu

Các dòng này không phù hợp với Market Basket Analysis vì chúng không phản ánh hành vi mua sản phẩm để cross-sell.

Do đó, ta loại bỏ các mã này trước khi xây dựng giỏ hàng và tính Product AOV.

In [ ]:
# Danh sách StockCode không phù hợp cho Market Basket Analysis
non_product_codes = [
    "POST",
    "DOT",
    "M",
    "m",
    "C2",
    "ADJUST",
    "ADJUST2",
    "BANK CHARGES",
    "D",
    "AMAZONFEE",
    "S",
    "B",
    "TEST001",
    "TEST002",
    "gift_0001_10",
    "gift_0001_20",
    "gift_0001_30",
    "gift_0001_40",
    "gift_0001_50",
    "gift_0001_70",
    "gift_0001_80"
]

# Kiểm tra số dòng sẽ bị loại bỏ
rows_to_remove = df[df["StockCode"].isin(non_product_codes)]

print("Rows to remove:", rows_to_remove.shape[0])

# Xem lại các mã sẽ bị loại bỏ
rows_to_remove[["StockCode", "Description"]].drop_duplicates().sort_values("StockCode")

Rows to remove: 4556


,StockCode,Description
71058,ADJUST,ADJUSTMENT BY JOHN ON 26/01/2010 17
70975,ADJUST,ADJUSTMENT BY JOHN ON 26/01/2010 16
249672,ADJUST2,ADJUSTMENT BY PETER ON JUN 25 2010
440698,AMAZONFEE,AMAZON FEE
825443,B,ADJUST BAD DEBT
18466,BANK CHARGES,BANK CHARGES
9292,C2,CARRIAGE
160443,D,DISCOUNT
2379,DOT,DOTCOM POSTAGE
2697,M,MANUAL


In [ ]:
# Loại bỏ các StockCode không phù hợp
df = df[~df["StockCode"].isin(non_product_codes)].copy()

# Kiểm tra kích thước dữ liệu sau khi loại bỏ
print("Shape after removing non-product StockCodes:", df.shape)

Shape after removing non-product StockCodes: (1003357, 9)


## Bước 11: Tạo cột LineTotal

Sau khi làm sạch dữ liệu sản phẩm, ta tạo cột `LineTotal` để tính giá trị của từng dòng giao dịch.

Công thức:

LineTotal = Quantity × UnitPrice

Cột này sẽ được dùng để tính doanh thu theo từng hóa đơn, AOV và các biến phục vụ hồi quy.



In [ ]:
# Tạo giá trị doanh thu cho từng dòng giao dịch
df["LineTotal"] = df["Quantity"] * df["UnitPrice"]

# Kiểm tra thống kê cơ bản của LineTotal
print(df["LineTotal"].describe())

# Xem thử vài dòng
df[["InvoiceNo", "StockCode", "Description", "Quantity", "UnitPrice", "LineTotal"]].head()

count    1.003357e+06
mean     1.957814e+01
std      1.999817e+02
min      1.000000e-03
25%      4.130000e+00
50%      1.008000e+01
75%      1.770000e+01
max      1.684696e+05
Name: LineTotal, dtype: float64


,InvoiceNo,StockCode,Description,Quantity,UnitPrice,LineTotal
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,6.95,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,6.75,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,6.75,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2.10,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,1.25,30.0


## Bước 12: Tạo basket và đặc trưng hóa đơn

Ở bước này, ta tạo hai bảng song song từ dữ liệu đã làm sạch:

1. `transactions`: chứa danh sách sản phẩm trong từng hóa đơn.
   - Dùng cho TransactionEncoder, Apriori và FP-Growth.

2. `invoice_features`: chứa các đặc trưng theo từng hóa đơn.
   - Dùng cho AOV, phân tích giỏ hàng và hồi quy.

Cách tách này giúp vừa giữ được danh sách sản phẩm để khai thác luật kết hợp, vừa giữ được thông tin doanh thu để phân tích tác động kinh doanh.

In [ ]:
# Tạo danh sách sản phẩm trong từng hóa đơn
transactions = (
    df.groupby("InvoiceNo")["StockCode"]
    .apply(lambda x: sorted(list(set(x))))
    .reset_index(name="Items")
)

# Tính số sản phẩm khác nhau trong từng hóa đơn
transactions["BasketSize"] = transactions["Items"].apply(len)

# Tạo đặc trưng doanh thu và thông tin hóa đơn
invoice_features = (
    df.groupby("InvoiceNo")
    .agg(
        ProductRevenue=("LineTotal", "sum"),
        TotalQuantity=("Quantity", "sum"),
        AvgUnitPrice=("UnitPrice", "mean"),
        CustomerID=("CustomerID", "first"),
        Country=("Country", "first")
    )
    .reset_index()
)

# Gộp basket với đặc trưng hóa đơn
basket_data = transactions.merge(
    invoice_features,
    on="InvoiceNo",
    how="left"
)

# Xem thử dữ liệu sau khi gộp
basket_data.head()

,InvoiceNo,Items,BasketSize,ProductRevenue,TotalQuantity,AvgUnitPrice,CustomerID,Country
0,489434,"[21232, 21523, 21871, 22041, 22064, 79323P, 79...",8,505.30,166,4.081250,13085.0,United Kingdom
1,489435,"[22195, 22349, 22350, 22353]",4,145.80,60,2.625000,13085.0,United Kingdom
2,489436,"[21181, 21333, 21754, 21755, 21756, 22107, 221...",19,630.33,193,3.730526,13078.0,United Kingdom
3,489437,"[10002, 20695, 20703, 20971, 21351, 21352, 213...",23,310.75,145,3.628261,15362.0,United Kingdom
4,489438,"[20711, 21033, 21100, 21252, 21329, 21410, 214...",17,2286.24,826,2.591176,18102.0,United Kingdom


## Bước 13: Loại bỏ giỏ hàng chỉ có một sản phẩm cho Rule Mining

Association Rule Mining cần ít nhất hai sản phẩm trong cùng một giỏ hàng để tạo luật kết hợp.

Các giỏ hàng chỉ có một sản phẩm không thể tạo ra luật dạng `A → B`.

Vì vậy, ta tạo một tập dữ liệu riêng cho Rule Mining bằng cách chỉ giữ các giỏ hàng có từ hai sản phẩm trở lên.

Lưu ý: bước này chỉ áp dụng cho Rule Mining, không áp dụng cho phân tích AOV hoặc hồi quy.

In [ ]:
# Chỉ giữ các giỏ hàng có từ 2 sản phẩm khác nhau trở lên
transactions_rules = transactions[
    transactions["BasketSize"] >= 2
].copy()

# Kiểm tra số lượng giỏ hàng trước và sau khi lọc
print("Total baskets:", transactions.shape[0])
print("Baskets for rule mining:", transactions_rules.shape[0])

# Xem thử vài giỏ hàng dùng cho rule mining
transactions_rules.head()

Total baskets: 39516
Baskets for rule mining: 36308


,InvoiceNo,Items,BasketSize
0,489434,"[21232, 21523, 21871, 22041, 22064, 79323P, 79...",8
1,489435,"[22195, 22349, 22350, 22353]",4
2,489436,"[21181, 21333, 21754, 21755, 21756, 22107, 221...",19
3,489437,"[10002, 20695, 20703, 20971, 21351, 21352, 213...",23
4,489438,"[20711, 21033, 21100, 21252, 21329, 21410, 214...",17


## Bước 14: Kiểm tra và lọc outlier BasketSize

Trước khi mã hóa transaction matrix, ta cần kiểm tra phân phối kích thước giỏ hàng.

Một số hóa đơn có thể chứa số lượng sản phẩm bất thường rất lớn.  
Các giỏ hàng quá lớn có thể làm tăng số lượng kết hợp sản phẩm, khiến Apriori hoặc FP-Growth chạy chậm và tạo ra nhiều luật nhiễu.

Vì vậy, ta kiểm tra phân phối BasketSize và loại bỏ các giỏ hàng quá lớn nếu cần.

In [ ]:
# Kiểm tra phân phối BasketSize
print(transactions_rules["BasketSize"].describe())

# Xem các mốc phân vị cao
print(transactions_rules["BasketSize"].quantile([0.90, 0.95, 0.99, 0.995, 0.999]))

count    36308.000000
mean        27.234990
std         43.454934
min          2.000000
25%          8.000000
50%         17.000000
75%         30.000000
max       1108.000000
Name: BasketSize, dtype: float64
0.900     53.000
0.950     76.000
0.990    203.000
0.995    298.465
0.999    539.386
Name: BasketSize, dtype: float64


## Bước 15: Loại bỏ outlier BasketSize

Phân phối BasketSize cho thấy có một số hóa đơn có số lượng sản phẩm rất lớn.

Giỏ hàng lớn bất thường có thể đại diện cho đơn bán buôn, đơn nhập liệu đặc biệt hoặc hành vi không phổ biến.  
Các giỏ hàng này có thể làm tăng số lượng tổ hợp sản phẩm và tạo ra nhiều luật nhiễu.

Vì vậy, ta loại bỏ các basket có kích thước vượt quá phân vị 99%.

In [ ]:
# Xác định ngưỡng outlier theo phân vị 99%
basket_size_threshold = transactions_rules["BasketSize"].quantile(0.99)

print("Basket size threshold:", basket_size_threshold)

# Chỉ giữ các giỏ hàng có BasketSize nhỏ hơn hoặc bằng ngưỡng
transactions_rules_filtered = transactions_rules[
    transactions_rules["BasketSize"] <= basket_size_threshold
].copy()

print("Baskets before outlier filtering:", transactions_rules.shape[0])
print("Baskets after outlier filtering:", transactions_rules_filtered.shape[0])

# Kiểm tra lại phân phối BasketSize sau khi lọc
print(transactions_rules_filtered["BasketSize"].describe())

Basket size threshold: 203.0
Baskets before outlier filtering: 36308
Baskets after outlier filtering: 35945
count    35945.000000
mean        23.976353
std         25.678631
min          2.000000
25%          8.000000
50%         17.000000
75%         29.000000
max        203.000000
Name: BasketSize, dtype: float64


## Bước 15: Mã hóa Transaction Matrix

Sau khi hoàn thành việc làm sạch dữ liệu và loại bỏ các basket bất thường, ta chuyển dữ liệu giao dịch sang dạng Transaction Matrix.

Trong Transaction Matrix:

- Mỗi dòng đại diện cho một hóa đơn (Invoice)
- Mỗi cột đại diện cho một sản phẩm (StockCode)
- Giá trị True nghĩa là sản phẩm xuất hiện trong hóa đơn
- Giá trị False nghĩa là sản phẩm không xuất hiện trong hóa đơn

Đây là định dạng đầu vào bắt buộc cho Apriori và FP-Growth.

In [ ]:
# Import TransactionEncoder
from mlxtend.preprocessing import TransactionEncoder

# Khởi tạo encoder
te = TransactionEncoder()

# Chuyển danh sách sản phẩm thành ma trận True/False
transaction_array = te.fit(
    transactions_rules_filtered["Items"]
).transform(
    transactions_rules_filtered["Items"]
)

# Chuyển thành DataFrame
transaction_matrix = pd.DataFrame(
    transaction_array,
    columns=te.columns_
)

print("Transaction matrix shape:")
print(transaction_matrix.shape)

transaction_matrix.head()

Transaction matrix shape:
(35945, 4810)


,10002,10002R,10080,10109,10120,10123C,10123G,10124A,10124G,10125,...,DCGS0058,DCGS0062,DCGS0066N,DCGS0069,DCGS0070,DCGS0076,DCGSSBOY,DCGSSGIRL,PADS,SP1002
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,True,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [ ]:
df.to_csv("online_retail_ii_basket_items.csv", index=False)

print("Saved: online_retail_ii_basket_items.csv")
display(df.head())

Saved: online_retail_ii_basket_items.csv


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,StockCodeLength,LineTotal
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,5,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,6,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,6,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,5,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,5,30.0


In [ ]:
# # 2. Save basket dataframe
# basket_data.to_csv("online_retail_ii_basket_df.csv", index=False)

# # 3. Save transaction matrix
# transaction_matrix.to_csv("online_retail_ii_transaction_matrix.csv", index=False)